In [1]:
import torch
import numpy as np
import normflows as nf
import pandas as pd

seed = 10
torch.manual_seed(seed)
torch.no_grad()

import sys
import os
c_directory = os.getcwd()
sys.path.append(os.path.join(c_directory, 'FCYeast2'))

from matplotlib import pyplot as plt
import FCYeast2_simulator
import architecture

enable_cuda = True
CUDA_LAUNCH_BLOCKING=1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/pessoa/Codes/SBI-Final/FC-inference/FCYeast2/FCYeast_simulator.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  autofluo.load_state_dict(torch.load('autofluorescen

In [2]:
dils_str = ['12','23']
dils = [.12,.23]
dfs = [pd.read_csv(os.path.dirname(c_directory) +'/clean_data/complete_d={}.csv'.format(d)) for d in dils]
x = [np.log(df['FL1-A'].to_numpy()) for df in dfs]

In [3]:
Nbins = 100

h,bins=[],[]
for xi in x:
    hi,bi = np.histogram(xi,bins=Nbins)
    h.append(torch.tensor(hi).to(device))
    bins.append(torch.tensor(bi).to(device))

TypeError: `bins` must be an integer, a string, or an array

In [ ]:
default_means =  np.array((10.,-1.,1. ,-2.3))[[0,1,2,3,1,2,3]]
default_sigmas = np.array(( 3.,1.5,1.5,1.0 ))[[0,1,2,3,1,2,3]]
target = FCYeast2_simulator.target()

/home/pessoa/Codes/SBI-Final/FC-inference/FCYeast2/FCYeast2_simulator.py:48: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.prior = torch.distributions.MultivariateNormal(torch.tensor(means).clone().detach().to(device), torch.diag(torch.tensor(sigmas)**2).clone().detach().to(device))
/home/pessoa/Codes/SBI-Final/FC-inference/FCYeast2/FCYeast2_simulator.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.params_dist = torch.distributions.MultivariateNormal(torch.tensor(means).clone().detach().to(device), torch.diag(torch.tensor(sigmas)**2).clone().detach().to(device))


In [ ]:
def logprior(params):
    z = (params-default_means)/default_sigmas
    return (- (np.power(z,2)/2) - np.log(default_sigmas) ).sum()

In [ ]:
def histogram_gpu(data,bins):
    #function needed bc torch.histogram is not working in cuda
    bins = bins.reshape(-1,1)
    arr = torch.logical_and(data>bins[:-1],data<bins[1:])
    return arr.sum(axis=1)

In [ ]:
eps = float(1/Nbins) #to prevent log of 0. sums to 1, as we had one extra datapoint spread into all bins
        
def ABC_log_likelihood(param,N=2**10):
    #simulate
    params_arbitrary = FCYeast2_simulator.transform_to_arbitrary(torch.tensor(param).to(device).float())
    
    simulations = target.sample(params_arbitrary,N=N)
    ll = []
    for (hi,bi,si) in zip(h,bins,simulations):
        counts = histogram_gpu(si,bins=bi) 
        counts += eps
        pi = counts/(N+1) #turn into probabilities
        ll.append( (hi*torch.log(pi)).sum() )
    return ll
               
def ABC_log_post(params,lprior=logprior):
    return sum([lp.sum() for lp in ABC_log_likelihood(params)]) + lprior(params)

In [ ]:
params_1k = default_means  + default_sigmas*np.random.normal(size=(1000,7))
best_param = default_means
lp_max = ABC_log_post(best_param)


for i in range(1,11):
    print(i)
    for par in (1/i)*(params_1k-best_param) + best_param:
        lp_par = ABC_log_post(par)
        if lp_par>=lp_max:
            best_param = par
            lp_max=lp_par
            print(best_param,lp_max)
            
del params_1k

RuntimeError: result type Float can't be cast to the desired output type Long

In [ ]:
np.histogram(x[1],bins[0].cpu().numpy())[0]

(array([ 172,  256,  279,  346,  418,  571,  775,  943, 1113, 1476, 1729,
        2043, 2433, 2948, 3369, 3633, 4060, 4378, 4521, 4672, 4702, 4450,
        4328, 3948, 3577, 3366, 2978, 2665, 2250, 2017, 1876, 1678, 1647,
        1592, 1450, 1313, 1286, 1236, 1146, 1148,  975,  989,  977, 1003,
         956,  970,  867,  918,  912,  829,  803,  796,  796,  764,  774,
         705,  691,  694,  647,  607,  587,  629,  594,  562,  562,  560,
         557,  501,  511,  439,  465,  409,  442,  425,  353,  369,  351,
         308,  279,  283,  244,  194,  195,  159,  110,   87,   66,   45,
          29,   12,    1,    0,    0,    0,    0,    0,    0,    0,    0,
           0]),
 array([ 6.05443935,  6.11660897,  6.17877859,  6.24094821,  6.30311783,
         6.36528745,  6.42745707,  6.4896267 ,  6.55179632,  6.61396594,
         6.67613556,  6.73830518,  6.8004748 ,  6.86264442,  6.92481404,
         6.98698367,  7.04915329,  7.11132291,  7.17349253,  7.23566215,
         7.29783177,  7.36